# 08 · Case Study: Healthcare Claims

> **Module 3 · Case Studies · Lecture 3.2**

In this notebook you will:

- Parse a nested JSON schema with `StructType` and `F.from_json`
- Explode diagnosis codes into one row per code
- Validate claims and route failures to a quarantine dataset
- Aggregate spend by provider, month, and diagnosis
- Detect 30-day spend outliers with a rolling z-score window function

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · The business problem

Healthcare claims arrive as JSON documents with nested structure: a claim header, a list
of diagnosis codes, and provider metadata. The analytics team needs to:

1. Parse the nested payload with an explicit schema.
2. Explode the diagnosis array into one row per diagnosis code.
3. Quarantine claims that fail validation instead of dropping them.
4. Aggregate spend by provider, month, and diagnosis.
5. Flag providers whose monthly spend is a 30-day statistical outlier.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
import json
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType, StructType, StructField, StringType, IntegerType,
    DoubleType, DateType, TimestampType,
)

DATA_DIR = "/tmp/dataproc_notebook_08_healthcare"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

SEED = 7
N_CLAIMS = 1200

# --- The nested schema ---------------------------------------------------
diagnosis_schema = StructType([
    StructField("code", StringType(), True),
    StructField("description", StringType(), True),
])

provider_schema = StructType([
    StructField("provider_npi", StringType(), True),
    StructField("provider_name", StringType(), True),
    StructField("specialty", StringType(), True),
])

claim_payload_schema = StructType([
    StructField("claim_id", StringType(), True),
    StructField("patient_id", StringType(), True),
    StructField("service_date", StringType(), True),
    StructField("billed_amount", DoubleType(), True),
    StructField("paid_amount", DoubleType(), True),
    StructField("diagnoses", ArrayType(diagnosis_schema), True),
    StructField("provider", provider_schema, True),
])

print("Nested claim schema:")
print(claim_payload_schema.simpleString())
print()
print("As JSON (this is what you persist to version control):")
print(claim_payload_schema.json())

## 2 · Generate sample claims

Each claim carries a nested `provider` struct, a `diagnoses` array, and a JSON-in-a-string
payload. We deliberately inject a few bad rows (missing amounts, negative spend) so the
quarantine path is exercised.

In [ ]:
claims = (
    spark.range(N_CLAIMS)
    .withColumn("claim_id", F.concat(F.lit("clm-"), F.col("id").cast("string")))
    .withColumn("patient_id", F.concat(F.lit("pat-"), (F.col("id") % 200).cast("string")))
    # NOTE: the provider and the service date must be assigned INDEPENDENTLY.
    #
    # `id % 120` for the date and `id % 15` for the provider are correlated
    # (15 divides 120), so a provider only ever books on every 15th day: each
    # provider ends up with 8 rows and a 30-row rolling window never fills.
    #
    # The fix is a hash whose multiplier is coprime with the modulus, applied
    # to DIFFERENT multipliers for the provider and the date. `xxhash64` is a
    # good mix, and using two different seeds keeps the two assignments
    # statistically independent.
    .withColumn("service_date", F.date_add(F.lit("2026-01-01").cast("date"),
                                           F.pmod(F.xxhash64(F.col("id"), F.lit(7)), 150)
                                             .cast("int"))
                            .cast("string"))
    .withColumn("billed_amount", F.round(F.rand(SEED) * 8000 + 50, 2))
    .withColumn("paid_amount", F.round(F.rand(SEED + 1) * 6000 + 40, 2))
    # --- Injected data-quality problems, so the quarantine path is exercised ---
    # every 97th claim: the paid amount went missing (null)
    # every 89th claim: a negative payment (a billing correction gone wrong)
    # every 131st claim: the service date is missing
    .withColumn(
        "paid_amount",
        F.when(F.col("id") % 97 == 0, F.lit(None).cast("double"))
         .when(F.col("id") % 89 == 0, F.lit(-250.0))
         .otherwise(F.col("paid_amount")),
    )
    .withColumn(
        "service_date",
        F.when(F.col("id") % 131 == 0, F.lit(None).cast("string"))
         .otherwise(F.col("service_date")),
    )
    .withColumn(
        "diagnoses",
        F.array(
            F.struct(
                F.concat(F.lit("DX"), (F.col("id") % 12).cast("string")).alias("code"),
                F.concat(F.lit("Diagnosis "), (F.col("id") % 12).cast("string")).alias("description"),
            ),
            F.struct(
                F.concat(F.lit("DX"), ((F.col("id") + 5) % 12).cast("string")).alias("code"),
                F.concat(F.lit("Comorbidity "), ((F.col("id") + 5) % 12).cast("string")).alias("description"),
            ),
        ),
    )
    .withColumn(
        "provider",
        F.struct(
            F.concat(F.lit("npi-"),
                     F.pmod(F.xxhash64(F.col("id"), F.lit(3)), 15).cast("string")
                     ).alias("provider_npi"),
            F.concat(F.lit("Provider "),
                     F.pmod(F.xxhash64(F.col("id"), F.lit(3)), 15).cast("string")
                     ).alias("provider_name"),
            F.concat(F.lit("Specialty "),
                     F.pmod(F.xxhash64(F.col("id"), F.lit(5)), 4).cast("string")
                     ).alias("specialty"),
        ),
    )
)

# Serialize the structured row to a JSON string -> "JSON in a string" column.
claims_json = (
    claims
    .select(
        F.col("claim_id"),
        F.to_json(
            F.struct(
                F.col("claim_id").alias("claim_id"),
                F.col("patient_id").alias("patient_id"),
                F.col("service_date").alias("service_date"),
                F.col("billed_amount").alias("billed_amount"),
                F.col("paid_amount").alias("paid_amount"),
                F.col("diagnoses").alias("diagnoses"),
                F.col("provider").alias("provider"),
            )
        ).alias("payload"),
    )
)

claims_json.show(3, truncate=False)
print("claims generated:", claims_json.count())

## 3 · Parse with `F.from_json`

`F.from_json(col, schema)` parses a JSON string into a nested struct using an explicit
`StructType`.

**Note:** `from_json` returns **null for the whole struct** when the JSON cannot be parsed —
so a row-level parse failure is indistinguishable from a genuinely null payload unless
you also check the raw string. The validation section below handles this.

In [ ]:
parsed = (
    claims_json
    .withColumn("parsed", F.from_json(F.col("payload"), claim_payload_schema))
)
parsed.select("claim_id", "parsed").show(3, truncate=False)

print("payload null count:", parsed.filter(F.col("parsed").isNull()).count())
print("parsed schema:")
parsed.select("parsed").printSchema()

## 4 · Flatten and explode the diagnosis array

`explode` turns one array element into one row. Use `explode_outer` to keep rows whose
array is null (with a null element) — usually what you want, because a claim with no
diagnosis is still a claim.

In [ ]:
# Flatten the nested provider struct with dot notation.
flat = (
    parsed
    .filter(F.col("parsed").isNotNull())
    .select(
        F.col("claim_id"),
        F.col("parsed.patient_id").alias("patient_id"),
        F.col("parsed.service_date").alias("service_date"),
        F.col("parsed.billed_amount").alias("billed_amount"),
        F.col("parsed.paid_amount").alias("paid_amount"),
        F.col("parsed.provider.provider_npi").alias("provider_npi"),
        F.col("parsed.provider.provider_name").alias("provider_name"),
        F.col("parsed.provider.specialty").alias("specialty"),
        F.col("parsed.diagnoses").alias("diagnoses"),
    )
)

# explode -> one row per diagnosis code.
exploded = flat.select(
    F.col("claim_id"),
    F.col("patient_id"),
    F.to_date(F.col("service_date")).alias("service_date"),
    F.col("billed_amount"),
    F.col("paid_amount"),
    F.col("provider_npi"),
    F.col("provider_name"),
    F.col("specialty"),
    F.explode_outer("diagnoses").alias("diagnosis"),
).withColumn(
    "diagnosis_code", F.col("diagnosis.code")
).withColumn(
    "diagnosis_description", F.col("diagnosis.description")
).drop("diagnosis")

print("claim rows :", flat.count())
print("exploded   :", exploded.count())
exploded.select(
    "claim_id", "service_date", "provider_name", "diagnosis_code", "paid_amount"
).show(8, truncate=False)

## 5 · Validation and quarantine

A claim is **valid** when it has an identifier, a service date, a non-null positive paid
amount, and at least one diagnosis. Everything else goes to quarantine with a reason.

**Note:** **Quarantine, never drop.** A rejected claim is a billing incident someone has to
resolve.

In [ ]:
from pyspark.sql import Window

validated = exploded.withColumn(
    "rejection_reason",
    F.when(F.col("claim_id").isNull(), F.lit("MISSING_CLAIM_ID"))
     .when(F.col("service_date").isNull(), F.lit("MISSING_SERVICE_DATE"))
     .when(F.col("paid_amount").isNull(), F.lit("MISSING_PAID_AMOUNT"))
     .when(F.col("paid_amount") < 0, F.lit("NEGATIVE_PAID_AMOUNT"))
     .when(F.col("diagnosis_code").isNull(), F.lit("MISSING_DIAGNOSIS_CODE"))
     .otherwise(F.lit(None)),
)

# A claim is rejected if any of its exploded rows is invalid.
claim_level = (
    validated
    .groupBy("claim_id")
    .agg(
        F.max(F.col("rejection_reason")).alias("rejection_reason"),
        F.count("*").alias("diagnosis_rows"),
    )
)
invalid_claims = claim_level.filter(F.col("rejection_reason").isNotNull()).select("claim_id")
valid_claim_ids = claim_level.filter(F.col("rejection_reason").isNull()).select("claim_id")

valid = (
    validated
    .join(valid_claim_ids, "claim_id", "inner")
    .drop("rejection_reason")
)
quarantine = (
    validated
    .join(invalid_claims, "claim_id", "inner")
    .filter(F.col("rejection_reason").isNotNull())
    .dropDuplicates(["claim_id"])
)

print("valid claims     :", valid_claim_ids.count())
print("quarantined      :", invalid_claims.count())
quarantine.select("claim_id", "rejection_reason").show(10, truncate=False)

# Write the quarantine to a dead-letter path.
quarantine_path = f"{DATA_DIR}/quarantine/claims"
quarantine.select("claim_id", "rejection_reason") \
    .write.mode("overwrite").json(quarantine_path)
print("quarantine written to", quarantine_path)

## 6 · Spend aggregation

Aggregate paid spend by provider, month, and diagnosis.

In [ ]:
spend = (
    valid
    .withColumn("service_month", F.date_format(F.col("service_date"), "yyyy-MM"))
    .groupBy("service_month", "provider_npi", "provider_name", "specialty", "diagnosis_code")
    .agg(
        F.countDistinct("claim_id").alias("claim_count"),
        F.countDistinct("patient_id").alias("patient_count"),
        F.round(F.sum("paid_amount"), 2).alias("total_paid"),
        F.round(F.sum("billed_amount"), 2).alias("total_billed"),
        F.round(F.avg("paid_amount"), 2).alias("avg_paid"),
    )
)

print("spend rows:", spend.count())
spend.orderBy(F.desc("total_paid")).show(10, truncate=False)

## 7 · 30-day rolling z-score outlier detection

For each provider, compute the daily paid spend, then a 30-day rolling mean and standard
deviation, and finally the z-score of each day:

```
z = (x - rolling_mean) / rolling_stddev
```

A |z| above 3 is an outlier.

**Caution:** A 30-day rolling window needs a **full history** to be meaningful, so the first 29
days have no z-score. Use `rowsBetween(-29, 0)` for a 30-row window including the
current row.

**Caution:** Division by zero: when the rolling standard deviation is 0 (a constant series), the
z-score is undefined. Guard it with `F.when(stddev > 0, ...)` and `otherwise(0.0)`.

The sample data books a **deliberate spike** on each provider's 30th and 31st service day, so
the detector has something real to find: daily spend jumps by ~400,000 against a rolling mean
of ~20,000, giving a z-score of about 5.3 — comfortably past the threshold of 3.

In [ ]:
# 1. Daily spend per provider.
#
# Injected anomaly.
#
# The spike is attached to provider-DAYS that are guaranteed to exist. We first
# collapse to one row per provider-day, number each provider's days in date
# order, and pick the 30th and 31st day of every provider. That lands the
# spike in the middle of the provider's history, so the 30-day rolling window
# sees both the history AND the spike when it is evaluated.
provider_days = (
    valid
    .groupBy("provider_npi", "provider_name", "service_date")
    .agg(F.round(F.sum("paid_amount"), 2).alias("base_spend"))
)

spike_days = (
    provider_days
    .withColumn(
        "day_index",
        F.row_number().over(
            Window.partitionBy("provider_npi").orderBy("service_date")
        ),
    )
    .withColumn(
        "daily_spend",
        F.when(F.col("day_index").isin(30, 31),
               F.col("base_spend") + F.lit(400000.0))
         .otherwise(F.col("base_spend")),
    )
    .select("provider_npi", "provider_name", "service_date", "daily_spend")
)

daily_provider_spend = spike_days

# 2. Rolling mean and stddev over a 30-row window (29 days back + today).
rolling_window = Window.partitionBy("provider_npi").orderBy("service_date").rowsBetween(-29, 0)

with_rolling = (
    daily_provider_spend
    .withColumn("rolling_mean", F.avg("daily_spend").over(rolling_window))
    .withColumn("rolling_stddev", F.stddev("daily_spend").over(rolling_window))
    .withColumn("rolling_count", F.count("daily_spend").over(rolling_window))
    .withColumn(
        "z_score",
        F.when(
            (F.col("rolling_count") >= 30) & (F.col("rolling_stddev") > 0),
            (F.col("daily_spend") - F.col("rolling_mean")) / F.col("rolling_stddev"),
        ).otherwise(F.lit(0.0)),
    )
    .withColumn("is_outlier", F.abs(F.col("z_score")) > F.lit(3.0))
)

print("provider-days:", with_rolling.count())
with_rolling.orderBy(F.desc("z_score")).show(10, truncate=False)

outliers = with_rolling.filter(F.col("is_outlier"))
print("\noutlier provider-days:", outliers.count())
outliers.select("provider_name", "service_date", "daily_spend", "rolling_mean", "z_score") \
        .orderBy(F.desc("z_score")).show(10, truncate=False)

### Optional: write to Parquet and BigQuery

Uncomment the BigQuery block after authenticating.

In [ ]:
out_dir = f"{DATA_DIR}/healthcare"
spend.write.mode("overwrite").option("compression", "snappy").partitionBy("service_month").parquet(
    f"{out_dir}/spend"
)
with_rolling.write.mode("overwrite").option("compression", "snappy").parquet(
    f"{out_dir}/provider_daily_rolling"
)

print("spend rows written:", spark.read.parquet(f"{out_dir}/spend").count())
print("rolling rows written:", spark.read.parquet(f"{out_dir}/provider_daily_rolling").count())

# ------------------------------------------------------------------
# OPTIONAL — BigQuery write.
# ------------------------------------------------------------------
# BQ_PROJECT = "your-project-id"
# (
#     spend.write
#     .format("bigquery")
#     .option("writeDisposition", "WRITE_TRUNCATE")
#     .option("temporaryGcsBucket", "your-staging-bucket")
#     .option("writeMethod", "direct")
#     .save(f"{BQ_PROJECT}.healthcare.provider_spend")
# )

print("BigQuery write cell is commented out — no GCP account required for this notebook.")

## 🔑 Key Takeaways

- `StructType` with `ArrayType` and nested `StructField`s describes a nested JSON document precisely.
- `F.from_json(col, schema)` parses a JSON string into a struct — but returns **null for the whole struct** on a parse failure.
- `explode_outer` keeps rows whose array is null; plain `explode` drops them.
- Dot notation (`parsed.provider.provider_npi`) flattens nested structs without a join.
- **Quarantine invalid claims with a reason** — never drop them silently.
- A 30-day rolling window is `rowsBetween(-29, 0)`; the first 29 days have no score.
- Guard the z-score against a zero standard deviation, or a constant series produces `NaN`/`inf`.
- Aggregate to a monthly grain for reporting while keeping the daily rolling series for anomaly detection.

---

## 📚 Next steps

- Add a second quarantine reason for claims whose paid amount exceeds their billed amount.
- Continue to `09-performance-tuning.ipynb` for tuning and benchmarking.
